# ComfyUI on Paperspace (A6000)

This notebook is tailored for Paperspace Gradient notebooks with NVIDIA A6000 GPUs.


# Installation

## Environment
* GPU: NVIDIA A6000
* Workspace: /notebooks
* Persistent storage: /storage

This notebook keeps models in /storage and symlinks them into ComfyUI for persistence across sessions.


In [ ]:
%%time
# --- Parameters --- #

# If set to true, ComfyUI and ComfyUI Manager will update after install.
update_comfy = True
update_manager = True
install_sdxl_model = True

# SDXL model that will be saved to permanent storage. Used for text-to-image.
sdxl_model_url = 'https://civitai.com/api/download/models/238308?type=Model&format=SafeTensor&size=pruned&fp=fp16'
sdxl_model_name = 'AlbedoBase.safetensors'

# ------------------- #

from os import path
import os

pip = '/notebooks/venv/bin/pip'
working_folder = '/notebooks'
storage_folder = '/storage'
model_storage = f'{storage_folder}/comfyui-models'
temp_folder = '/tmp/comfyui-temp'

%cd /notebooks
if not path.exists('ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
if update_comfy:
    get_ipython().system('git pull')

!{pip} install -r requirements.txt

models_root = model_storage
checkpoints = f'{models_root}/checkpoints'
loras = f'{models_root}/loras'
custom_nodes = f'{working_folder}/ComfyUI/custom_nodes'
outputs = f'{working_folder}/ComfyUI/output'
zip_outputs = f'{working_folder}/outputs'

os.makedirs(temp_folder, exist_ok=True)
os.makedirs(checkpoints, exist_ok=True)
os.makedirs(loras, exist_ok=True)

# Link persistent model folders into ComfyUI
for name, src in {
    'checkpoints': checkpoints,
    'loras': loras,
}.items():
    dst = f'{working_folder}/ComfyUI/models/{name}'
    if path.islink(dst):
        os.unlink(dst)
    elif path.isdir(dst):
        get_ipython().system(f'rm -rf {dst}')
    get_ipython().system(f'ln -s {src} {dst}')

# Install the node manager
%cd $working_folder/ComfyUI/custom_nodes
if not path.exists('ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager
if update_manager:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

# Install initial models
%cd $checkpoints
if install_sdxl_model and not path.exists(f'{checkpoints}/{sdxl_model_name}'):
    get_ipython().system(f'wget -O "{sdxl_model_name}" "{sdxl_model_url}"')

# Install an initial LoRA
%cd $loras
model_url = 'https://civitai.com/api/download/models/137124?type=Model&format=SafeTensor'
model_name = 'DreamArt.safetensors'
if not path.exists(f'{loras}/{model_name}'):
    get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

# Frame interpolation nodes
%cd $custom_nodes
if not path.exists('ComfyUI-Frame-Interpolation'):
    !git clone https://github.com/Fannovel16/ComfyUI-Frame-Interpolation
%cd ComfyUI-Frame-Interpolation
!git pull
!python install.py

# Modded nodes, used for math in the text-to-video and image-to-video workflow
%cd $custom_nodes
if not path.exists('Derfuu_ComfyUI_ModdedNodes'):
    !git clone https://github.com/Derfuu/Derfuu_ComfyUI_ModdedNodes
%cd Derfuu_ComfyUI_ModdedNodes
!git pull

# Update PyTorch stack for A6000
!{pip} install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!{pip} install --upgrade numpy

# Pinggy script
%cd /notebooks
!wget https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /notebooks/pinggy.py


In [ ]:
# ============================================================
# SynthID Bypass - Complete Setup (Nodes + Models + Workflows)
# For Paperspace with A6000
# ============================================================

import os

pip = '/notebooks/venv/bin/pip'
model_storage = '/storage/comfyui-models'

# ------------------------------------------------------------
# 1. INSTALL CUSTOM NODES
# ------------------------------------------------------------
print("📦 Installing custom nodes...")

%cd /notebooks/ComfyUI/custom_nodes

# ComfyUI Impact Pack
if not os.path.exists('ComfyUI-Impact-Pack'):
    !git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git
    %cd ComfyUI-Impact-Pack
    !{pip} install -r requirements.txt
    %cd ..
    # Install Impact Pack submodules
    %cd ComfyUI-Impact-Pack
    !git submodule update --init --recursive
    %cd ..

# ComfyUI-dype
if not os.path.exists('ComfyUI-DyPE'):
    !git clone https://github.com/wildminder/ComfyUI-DyPE.git

# rgthree-comfy
if not os.path.exists('rgthree-comfy'):
    !git clone https://github.com/rgthree/rgthree-comfy.git

# Masquerade Nodes
if not os.path.exists('masquerade-nodes-comfyui'):
    !git clone https://github.com/BadCafeCode/masquerade-nodes-comfyui.git

# ComfyUI-Inpaint-CropAndStitch
if not os.path.exists('ComfyUI-Inpaint-CropAndStitch'):
    !git clone https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch.git

# SeedVR2 VideoUpscaler Nodes
if not os.path.exists('ComfyUI-SeedVR2_VideoUpscaler'):
    !git clone https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git
    %cd ComfyUI-SeedVR2_VideoUpscaler
    !{pip} install -r requirements.txt
    %cd ..

print("✅ Custom nodes installed!")

# ------------------------------------------------------------
# 2. CREATE MODEL DIRECTORIES
# ------------------------------------------------------------
print("\n📁 Creating model directories...")

model_dirs = [
    'vae',
    'diffusion_models',
    'text_encoders',
    'sams',
    'model_patches',
    'ultralytics/bbox',
    'SEEDVR2'
]

for d in model_dirs:
    os.makedirs(f'{model_storage}/{d}', exist_ok=True)

print("✅ Directories created!")

# ------------------------------------------------------------
# 3. DOWNLOAD ALL MODELS
# ------------------------------------------------------------
print("\n⬇️ Downloading models (this may take a while)...")

# Z-Image Turbo VAE
if not os.path.exists(f'{model_storage}/vae/ae.safetensors'):
    print("  -> Downloading VAE...")
    !wget -q --show-progress -O {model_storage}/vae/ae.safetensors \
        https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors

# Z-Image Turbo Diffusion Model
if not os.path.exists(f'{model_storage}/diffusion_models/z_image_turbo_bf16.safetensors'):
    print("  -> Downloading Diffusion Model...")
    !wget -q --show-progress -O {model_storage}/diffusion_models/z_image_turbo_bf16.safetensors \
        https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors

# Qwen Text Encoder
if not os.path.exists(f'{model_storage}/text_encoders/qwen_3_4b.safetensors'):
    print("  -> Downloading Text Encoder...")
    !wget -q --show-progress -O {model_storage}/text_encoders/qwen_3_4b.safetensors \
        https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors

# SAM Model
if not os.path.exists(f'{model_storage}/sams/sam_vit_b_01ec64.pth'):
    print("  -> Downloading SAM Model...")
    !wget -q --show-progress -O {model_storage}/sams/sam_vit_b_01ec64.pth \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

# ControlNet Union
if not os.path.exists(f'{model_storage}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union.safetensors'):
    print("  -> Downloading ControlNet...")
    !wget -q --show-progress -O {model_storage}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union.safetensors \
        https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union/resolve/main/Z-Image-Turbo-Fun-Controlnet-Union.safetensors

# YOLO Face Detection
if not os.path.exists(f'{model_storage}/ultralytics/bbox/yolov8n-face.pt'):
    print("  -> Downloading YOLO Face Model...")
    !wget -q --show-progress -O {model_storage}/ultralytics/bbox/yolov8n-face.pt \
        https://huggingface.co/deepghs/yolo-face/resolve/739664f2d00e436a8882238f83175ab0f6497578/yolov8n-face/model.pt

# SeedVR2 VAE
if not os.path.exists(f'{model_storage}/SEEDVR2/ema_vae_fp16.safetensors'):
    print("  -> Downloading SeedVR2 VAE...")
    !wget -q --show-progress -O {model_storage}/SEEDVR2/ema_vae_fp16.safetensors \
        https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/ema_vae_fp16.safetensors

# SeedVR2 7B Sharp Model (best quality for A6000)
if not os.path.exists(f'{model_storage}/SEEDVR2/seedvr2_ema_7b_sharp_fp16.safetensors'):
    print("  -> Downloading SeedVR2 7B Sharp Model (16.5GB - this will take a while)...")
    !wget -q --show-progress -O {model_storage}/SEEDVR2/seedvr2_ema_7b_sharp_fp16.safetensors \
        https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/seedvr2_ema_7b_sharp_fp16.safetensors

print("✅ All models downloaded!")

# ------------------------------------------------------------
# 4. LINK MODEL DIRECTORIES TO COMFYUI
# ------------------------------------------------------------
print("\n🔗 Linking model directories to ComfyUI...")

link_dirs = ['vae', 'diffusion_models', 'text_encoders', 'sams', 'model_patches', 'ultralytics', 'SEEDVR2']

for model_dir in link_dirs:
    src = f'{model_storage}/{model_dir}'
    dst = f'/notebooks/ComfyUI/models/{model_dir}'
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.isdir(dst):
        get_ipython().system(f'rm -rf {dst}')
    get_ipython().system(f'ln -s {src} {dst}')

print("✅ Model directories linked!")

# ------------------------------------------------------------
# 5. DOWNLOAD WORKFLOW FILES
# ------------------------------------------------------------
print("\n📥 Downloading SynthID Bypass workflows...")

%cd /notebooks/ComfyUI

if not os.path.exists('/notebooks/ComfyUI/Synthid_Bypass.json'):
    !wget -q https://raw.githubusercontent.com/WulfhardMavuto/Synthid-Bypass/main/Synthid_Bypass.json

if not os.path.exists('/notebooks/ComfyUI/Synthid_Bypass_Portrait.json'):
    !wget -q https://raw.githubusercontent.com/WulfhardMavuto/Synthid-Bypass/main/Synthid_Bypass_Portrait.json

print("✅ Workflows downloaded!")

# ------------------------------------------------------------
# DONE!
# ------------------------------------------------------------
print("\n" + "="*60)
print("🎉 SETUP COMPLETE!")
print("="*60)
print("""
Next steps:
1. Run the WebUI cell to start ComfyUI
2. Open the Pinggy link in your browser
3. Drag & drop one of these workflows onto the canvas:
   - Synthid_Bypass.json (general purpose)
   - Synthid_Bypass_Portrait.json (for portraits)
4. Load your image and click 'Queue Prompt'

Workflows are saved in: /notebooks/ComfyUI/
Models are stored in:   /storage/comfyui-models/ (persistent)
""")


--- 
# WebUI

## Start the WebUI with Pinggy
* Wait for the GUI to start.  
* Click the link that ends with .pinggy.link 😁
* If generation is still running after the link expires in an hour, wait for the generation to complete and restart the WebUI code block to get a new link

In [ ]:
# Starting the Web UI with pinggy
%cd /notebooks/ComfyUI
!python /notebooks/pinggy.py --command='python /notebooks/ComfyUI/main.py --listen 0.0.0.0 --port 8188' --port=8188


## Start the WebUI with Zrok

### Install Zrok

In [ ]:
# Install Zrok (only needs to run once)

!mkdir -p /notebooks/zrok
%cd /notebooks/zrok
!rm -f zrok*.gz
!wget https://github.com/openziti/zrok/releases/download/v1.0.4/zrok_1.0.4_linux_amd64.tar.gz
!tar -xvf ./zrok*.gz
!chmod a+x /notebooks/zrok/zrok


### Create a Zrok account
Enter your email address in the email variable

In [ ]:
email = '####@gmail.com' # replace with your email

# --------------

cmd = '/notebooks/zrok/zrok invite'
log = '/notebooks/zrok/log.txt'

!{pip} install pexpect
!touch $log

import pexpect
import time
child = pexpect.spawn('bash')
child.sendline(f'{cmd} | tee {log}')
child.expect('enter and verify your email address:')
child.sendline(email)
child.expect(pexpect.EOF)

print('Check your email inbox for a verification link, then run the next cell.')


### Enable Zrok 
Paste your Zrok token in the token variable

In [ ]:
# Enable Zrok (needs to run once per instance)
# Paste your Zrok token in the token variable

token = ""
!chmod a+x /notebooks/zrok/zrok
!/notebooks/zrok/zrok enable $token


### Start the WebUI with Zrok

In [ ]:
# Start the WebUI with Zrok
%cd /notebooks/ComfyUI
command = 'python /notebooks/ComfyUI/main.py --listen 0.0.0.0 --port 8188'
port = '8188'
# ------------------------

!chmod a+x /notebooks/zrok/zrok
cmd = f'{command} & /notebooks/zrok/zrok share public http://localhost:{port} --headless'
get_ipython().system(cmd)


---
# Model Management

## Install a model

Copy the model URL to the model_url field. Make sure the model can be accessed publicly, without being signed into a website.

In [ ]:
# Install a model in permanent storage
# Models are stored in /storage/comfyui-models

model_url = 'https://civitai.com/api/download/models/160191?type=Model&format=SafeTensor&size=full&fp=fp16'
model_name = 'model.safetensors'

%cd /storage/comfyui-models/checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')


In [ ]:
# Install a LoRA in permanent storage
model_url = 'https://civitai.com/api/download/models/137124?type=Model&format=SafeTensor'
model_name = 'DreamArt.safetensors'

%cd /storage/comfyui-models/loras
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')


In [ ]:
# Install a model in temporary storage
#model_url = 'https://civitai.com/api/download/models/160191?type=Model&format=SafeTensor&size=full&fp=fp16'
#model_name = 'model.safetensors'

%cd /tmp/comfyui-temp
#get_ipython().system(f'wget -O "{model_name}" "{model_url}"')


## Download a model for a custom node

In [ ]:
model_folder = '/notebooks/ComfyUI/custom_nodes/my_node/models'
model_url = ''
model_name = 'model.safetensors'

%cd $model_folder
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')


---
# File Browser

## Install FileBrowser

In [ ]:
%cd /notebooks
!wget https://github.com/filebrowser/filebrowser/releases/download/v2.27.0/linux-amd64-filebrowser.tar.gz
!tar xvfz linux-amd64-filebrowser.tar.gz
!chmod a+x /notebooks/filebrowser
!/notebooks/filebrowser config init
!/notebooks/filebrowser config set --auth.method=noauth > /dev/null
!/notebooks/filebrowser users add admin admin


## Run FileBrowser

In [ ]:
%cd /notebooks
!chmod a+x /notebooks/filebrowser

!python /notebooks/pinggy.py --command='/notebooks/filebrowser -c "/notebooks/.filebrowser.json"' --port=8080


# 
# Delete a model

In [ ]:
# List permanent models
!ls -la /storage/comfyui-models/checkpoints

# Delete a model
model_to_delete = '/storage/comfyui-models/checkpoints/model.safetensors'
!rm $model_to_delete


In [ ]:
# Check the size of a model
!du -sh /storage/comfyui-models/loras/harrlogos.safetensors


# 
# Delete everything in the working folder

In [ ]:
# Delete the working folder
!rm -rf /notebooks/ComfyUI
